# Atividade 11 - Rede Neural ART-1

Este notebook apresenta a implementação e simulação de uma rede neural **ART-1 (Adaptive Resonance Theory 1)** para classificar e agrupar 10 situações de comportamento de um processo industrial baseadas em 16 variáveis de status binárias ($x_1, \dots, x_{16}$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class ART1:
    def __init__(self, input_dim=16, rho=0.5, L=2.0):
        self.input_dim = input_dim
        self.rho = rho
        self.L = L
        
        # Bottom-up weights
        self.W_bu = np.empty((0, input_dim))
        # Top-down weights
        self.W_td = np.empty((0, input_dim))
        
    def add_category(self, x):
        t_new = np.copy(x)
        norm_x = np.sum(x)
        b_new = (self.L * x) / (self.L - 1.0 + norm_x)
        
        self.W_td = np.vstack([self.W_td, t_new])
        self.W_bu = np.vstack([self.W_bu, b_new])
        return len(self.W_td) - 1

    def compute_activations(self, x):
        if len(self.W_bu) == 0:
            return np.array([])
        return np.dot(self.W_bu, x)

    def train(self, X, max_epochs=100):
        N = len(X)
        prev_classifications = np.full(N, -1)
        
        for epoch in range(max_epochs):
            classifications = np.full(N, -1)
            
            for idx, x in enumerate(X):
                norm_x = np.sum(x)
                y_uncommitted = (self.L * norm_x) / (self.L - 1.0 + self.input_dim)
                num_classes = len(self.W_bu)
                activations = self.compute_activations(x)
                active = np.ones(num_classes, dtype=bool)
                
                winner_idx = -1
                while True:
                    candidate_idx = -1
                    candidate_act = -1.0
                    for j in range(num_classes):
                        if active[j] and activations[j] > candidate_act:
                            candidate_act = activations[j]
                            candidate_idx = j
                            
                    if candidate_idx != -1 and candidate_act >= y_uncommitted:
                        t_j = self.W_td[candidate_idx]
                        intersection = np.logical_and(x, t_j).astype(float)
                        norm_intersection = np.sum(intersection)
                        
                        if norm_intersection / norm_x >= self.rho:
                            winner_idx = candidate_idx
                            self.W_td[winner_idx] = intersection
                            self.W_bu[winner_idx] = (self.L * intersection) / (self.L - 1.0 + norm_intersection)
                            break
                        else:
                            active[candidate_idx] = False
                    else:
                        winner_idx = self.add_category(x)
                        break
                classifications[idx] = winner_idx
                
            if np.array_equal(classifications, prev_classifications):
                break
            prev_classifications = np.copy(classifications)
            
        return prev_classifications

## Dados de Entrada
Definindo as 10 situações e as 16 variáveis de status binárias:

In [ ]:
SITUATIONS = np.array([
    [0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1], # Situação 1
    [1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0], # Situação 2
    [1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1], # Situação 3
    [1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0], # Situação 4
    [0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1], # Situação 5
    [1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1], # Situação 6
    [1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0], # Situação 7
    [1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1], # Situação 8
    [0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1], # Situação 9
    [0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1]  # Situação 10
])

## Função Auxiliar para Visualização
Criação dos gráficos de agrupamento/heatmap:

In [ ]:
def plot_clusters(X, labels, rho):
    unique_labels = sorted(list(set(labels)))
    sorted_indices = []
    for l in unique_labels:
        sorted_indices.extend([i for i, val in enumerate(labels) if val == l])
        
    X_sorted = X[sorted_indices]
    labels_sorted = [labels[i] for i in sorted_indices]
    sit_names_sorted = [f"Situação {i+1} (Classe {labels[i] + 1})" for i in sorted_indices]
    
    plt.figure(figsize=(12, 7), dpi=100)
    plt.imshow(X_sorted, cmap='Blues', aspect='auto', interpolation='nearest', vmin=0, vmax=1)
    
    plt.gca().set_xticks(np.arange(-.5, 16, 1), minor=True)
    plt.gca().set_yticks(np.arange(-.5, len(X), 1), minor=True)
    plt.grid(which='minor', color='gray', linestyle='-', linewidth=0.5)
    
    plt.title(f"Agrupamento ART-1 - Vigilância (rho = {rho})", fontsize=14, fontweight='bold', pad=15)
    plt.xlabel("Variáveis de Status (x1 a x16)", fontsize=12, labelpad=10)
    plt.ylabel("Situações", fontsize=12, labelpad=10)
    
    plt.xticks(np.arange(16), [f"x{i+1}" for i in range(16)])
    plt.yticks(np.arange(len(X)), sit_names_sorted)
    
    current_idx = 0
    for l in unique_labels:
        cnt = labels_sorted.count(l)
        if current_idx + cnt < len(X):
            plt.axhline(y=current_idx + cnt - 0.5, color='#d62728', linestyle='--', linewidth=2.0)
        current_idx += cnt
        
    cbar = plt.colorbar(ticks=[0, 1])
    cbar.ax.set_yticklabels(['0 (Inativo)', '1 (Ativo)'])
    plt.tight_layout()
    plt.show()

## Execução das Simulações
Rodando a classificação para $\rho = 0.5, 0.8, 0.9, 0.99$:

In [ ]:
vigilance_values = [0.5, 0.8, 0.9, 0.99]
for rho in vigilance_values:
    print(f"\n" + "="*50)
    print(f"Simulação com Vigilância rho = {rho}")
    print("="*50)
    
    art = ART1(input_dim=16, rho=rho, L=2.0)
    labels = art.train(SITUATIONS)
    
    num_classes = len(art.W_td)
    print(f"Quantidade de Classes Ativas: {num_classes}")
    
    clusters = {}
    for idx, lbl in enumerate(labels):
        cls_name = f"Classe {lbl + 1}"
        if cls_name not in clusters:
            clusters[cls_name] = []
        clusters[cls_name].append(f"Situação {idx + 1}")
        
    for cls_name, sits in sorted(clusters.items()):
        print(f"  {cls_name}: {', '.join(sits)}")
        
    plot_clusters(SITUATIONS, labels, rho)